# (1) Import libraries
- Notebook gốc sử dụng `scikit-learn` (`GaussianNB`) và `numpy` (không dùng Keras).
- Giữ nguyên toàn bộ phần gốc ở mục (1)–(4), đồng thời bổ sung mục (5) minh họa triển khai tương đương bằng **PyTorch Tensors**.

In [ ]:
from sklearn import datasets
from sklearn.naive_bayes import GaussianNB

import numpy as np
import time
import torch

# (2) Preparing Iris dataset

In [ ]:
iris = datasets.load_iris()

# get training and testing indexes randomly (60% train / 40% test)
train_indx = np.random.choice(range(0, 150), int(iris.target.shape[0] / 100 * 60), replace=False)
test_indx = np.array([indx for indx in np.arange(0, 150, 1) if indx not in train_indx])

x_train = iris.data[train_indx, :]
y_train = iris.target[train_indx]

x_test = iris.data[test_indx, :]
y_test = iris.target[test_indx]

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

# (3) Parameters estimation

In [ ]:
mu_0 = np.mean(x_train[y_train == 0], axis=0)
sigma_0 = np.cov(x_train[y_train == 0], rowvar=False)
print('Class 0 mean:', mu_0)
print('Class 0 cov:\n', sigma_0)

mu_1 = np.mean(x_train[y_train == 1], axis=0)
sigma_1 = np.cov(x_train[y_train == 1], rowvar=False)
print('Class 1 mean:', mu_1)
print('Class 1 cov:\n', sigma_1)

mu_2 = np.mean(x_train[y_train == 2], axis=0)
sigma_2 = np.cov(x_train[y_train == 2], rowvar=False)
print('Class 2 mean:', mu_2)
print('Class 2 cov:\n', sigma_2)

# (4) Bayesian decision making (scikit-learn GaussianNB)

In [ ]:
gnb = GaussianNB()

y_pred = gnb.fit(x_train, y_train).predict(x_test)
print('Testing classification accuracy (sklearn GaussianNB): %.4f' % ((y_test == y_pred).sum() / x_test.shape[0]))

# (5) PyTorch Tensor Implementation (Gaussian Naive Bayes)

In [ ]:
# Chuyển dữ liệu sang PyTorch Tensor
x_train_t = torch.tensor(x_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

classes = torch.unique(y_train_t)
log_priors = []
means = []
vars_ = []
eps = 1e-9

for c in classes:
    x_c = x_train_t[y_train_t == c]
    log_priors.append(torch.log(torch.tensor(len(x_c) / len(x_train_t), dtype=torch.float32)))
    means.append(x_c.mean(dim=0))
    vars_.append(x_c.var(dim=0, unbiased=False) + eps)

# Tính log-posterior cho từng mẫu test
log_posteriors = []
for i in range(len(classes)):
    dist = torch.distributions.Normal(means[i], torch.sqrt(vars_[i]))
    log_likelihood = dist.log_prob(x_test_t).sum(dim=1)
    log_posteriors.append(log_priors[i] + log_likelihood)

log_posteriors = torch.stack(log_posteriors, dim=1)
y_pred_torch = torch.argmax(log_posteriors, dim=1)

acc_torch = (y_pred_torch == y_test_t).float().mean().item()
print('Testing classification accuracy (PyTorch GaussianNB): %.4f' % acc_torch)